In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=False)

# Custom Dataset
class OpinionDataset(Dataset):
    def __init__(self, texts, level1, level2, level3, senti_scores, tokenizer, max_len=128):
        self.texts = texts
        self.level1 = level1
        self.level2 = level2
        self.level3 = level3
        self.senti_scores = senti_scores  # New feature
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.texts[idx], truncation=True, padding='max_length',
                                  max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label1': torch.tensor(self.level1[idx], dtype=torch.long),
            'label2': torch.tensor(-1 if pd.isna(self.level2[idx]) else self.level2[idx], dtype=torch.long),
            'label3': torch.tensor(-1 if pd.isna(self.level3[idx]) else self.level3[idx], dtype=torch.long),
            'senti_score': torch.tensor(self.senti_scores[idx], dtype=torch.float),  # New feature
            'text': self.texts[idx]
        }

# Hierarchical Model
class HierarchicalBertweet(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("vinai/bertweet-base")
        self.dropout = nn.Dropout(0.3)
        
        # New layer to process sentiment score
        self.senti_proj = nn.Linear(1, 64)  # Project 1D score to 64D
        
        # Modified classifiers to take both BERT features and sentiment features
        self.classifier1 = nn.Linear(768 + 64, 3)  # 768 (BERT) + 64 (senti)
        self.classifier2 = nn.Linear(768 + 64, 3)
        self.classifier3 = nn.Linear(768 + 64, 4)

    def forward(self, input_ids, attention_mask, senti_score):
        # Get BERT features
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)[0][:, 0]  # CLS token
        bert_features = self.dropout(bert_output)
        
        # Process sentiment score
        senti_features = self.senti_proj(senti_score.unsqueeze(1))  # Shape: (batch_size, 64)
        senti_features = torch.relu(senti_features)
        
        # Combine features
        combined_features = torch.cat([bert_features, senti_features], dim=1)
        
        # Classifiers
        out1 = self.classifier1(combined_features)
        out2 = self.classifier2(combined_features)
        out3 = self.classifier3(combined_features)
        return out1, out2, out3

def train_and_evaluate(df, tokenizer, epochs=3, batch_size=16, folds=10):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

    all_preds1, all_labels1 = [], []
    all_preds2, all_labels2 = [], []
    all_preds3, all_labels3 = [], []

    misclassified_data = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(df['text'], df['level1'])):
        print(f"\n====== Fold {fold+1} ======")
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]

        train_dataset = OpinionDataset(train_df['text'].tolist(), 
                                      train_df['level1'].tolist(),
                                      train_df['level2'].tolist(), 
                                      train_df['level3'].tolist(),
                                      train_df['senti_score'].tolist(), 
                                      tokenizer)
        val_dataset = OpinionDataset(val_df['text'].tolist(), 
                                    val_df['level1'].tolist(),
                                    val_df['level2'].tolist(), 
                                    val_df['level3'].tolist(),
                                    val_df['senti_score'].tolist(),  
                                    tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

        model = HierarchicalBertweet().cuda()
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
        loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
        best_score = 0.0
        best_model_path = f"best_model_fold{fold+1}.pt"
        for epoch in range(epochs):
            model.train()
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
                input_ids = batch['input_ids'].cuda()
                attention_mask = batch['attention_mask'].cuda()
                label1 = batch['label1'].cuda()
                label2 = batch['label2'].cuda()
                label3 = batch['label3'].cuda()
                senti_score = batch['senti_score'].cuda()

                out1, out2, out3 = model(input_ids, attention_mask, senti_score)
                loss1 = loss_fn(out1, label1)
                loss2 = loss_fn(out2, label2)
                loss3 = loss_fn(out3, label3)
                loss = loss1 + loss2 + loss3

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Evaluation after each epoch
            model.eval()
            val_true1, val_pred1 = [], []
            val_true2, val_pred2 = [], []
            val_true3, val_pred3 = [], []

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].cuda()
                    attention_mask = batch['attention_mask'].cuda()
                    label1 = batch['label1']
                    label2 = batch['label2']
                    label3 = batch['label3']
                    senti_score = batch['senti_score'].cuda()

                    out1, out2, out3 = model(input_ids, attention_mask, senti_score)
                    pred1 = torch.argmax(out1, dim=1).cpu()
                    pred2 = torch.argmax(out2, dim=1).cpu()
                    pred3 = torch.argmax(out3, dim=1).cpu()

                    val_true1.extend(label1.tolist())
                    val_pred1.extend(pred1.tolist())

                    for i in range(len(label1)):
                        if label1[i] == 2:
                            val_true2.append(label2[i].item())
                            val_pred2.append(pred2[i].item())
                            if label2[i] == 0:
                                val_true3.append(label3[i].item())
                                val_pred3.append(pred3[i].item())

            lev1_acc = accuracy_score(val_true1, val_pred1)
            lev1_f1 = f1_score(val_true1, val_pred1, average='weighted')
            combined_score = lev1_acc + lev1_f1

            print(f"\n📊 Epoch {epoch+1} Evaluation:")
            print(f"  Level 1 Accuracy: {lev1_acc:.4f}, F1: {lev1_f1:.4f}, Score: {combined_score:.4f}")
            if val_true2:
                print(f"  Level 2 Accuracy: {accuracy_score(val_true2, val_pred2):.4f}, F1: {f1_score(val_true2, val_pred2, average='weighted'):.4f}")
            if val_true3:
                print(f"  Level 3 Accuracy: {accuracy_score(val_true3, val_pred3):.4f}, F1: {f1_score(val_true3, val_pred3, average='weighted'):.4f}")

            if combined_score > best_score:
                best_score = combined_score
                torch.save(model.state_dict(), best_model_path)
                print(f"💾 Best model saved at Epoch {epoch+1} with score = {combined_score:.4f}")

        # Load best model and evaluate
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].cuda()
                attention_mask = batch['attention_mask'].cuda()
                senti_score = batch['senti_score'].cuda()
                label1 = batch['label1']
                label2 = batch['label2']
                label3 = batch['label3']
                texts = batch['text']

                out1, out2, out3 = model(input_ids, attention_mask, senti_score)
                pred1 = torch.argmax(out1, dim=1).cpu()
                pred2 = torch.argmax(out2, dim=1).cpu()
                pred3 = torch.argmax(out3, dim=1).cpu()

                for i in range(len(texts)):
                    all_preds1.append(pred1[i].item())
                    all_labels1.append(label1[i].item())

                    if label1[i] == 2:
                        all_preds2.append(pred2[i].item())
                        all_labels2.append(label2[i].item())

                        if label2[i] == 0:
                            all_preds3.append(pred3[i].item())
                            all_labels3.append(label3[i].item())

                            if label3[i].item() != pred3[i].item():
                                misclassified_data.append({
                                    "text": texts[i],
                                    "true_level1": label1[i].item(), "pred_level1": pred1[i].item(),
                                    "true_level2": label2[i].item(), "pred_level2": pred2[i].item(),
                                    "true_level3": label3[i].item(), "pred_level3": pred3[i].item()
                                })
                    elif label1[i].item() != pred1[i].item():
                        misclassified_data.append({
                            "text": texts[i],
                            "true_level1": label1[i].item(), "pred_level1": pred1[i].item()
                        })

    return (all_labels1, all_preds1,
            all_labels2, all_preds2,
            all_labels3, all_preds3,
            pd.DataFrame(misclassified_data),
            model)

def print_results(y_true1, y_pred1, y_true2, y_pred2, y_true3, y_pred3, misclassified_df):
    print("\n===== Level 1 Results =====")
    print(classification_report(y_true1, y_pred1, target_names=["NOISE", "OBJECTIVE", "SUBJECTIVE"]))
    print(confusion_matrix(y_true1, y_pred1))

    print("\n===== Level 2 Results (SUBJECTIVE only) =====")
    print(classification_report(y_true2, y_pred2, target_names=["NEUTRAL", "NEGATIVE", "POSITIVE"]))
    print(confusion_matrix(y_true2, y_pred2))

    print("\n===== Level 3 Results (NEUTRAL only) =====")
    print(classification_report(y_true3, y_pred3, target_names=["NEUTRAL", "QUESTIONS", "ADS", "MISC"]))
    print(confusion_matrix(y_true3, y_pred3))

    print(f"\nTotal misclassified examples: {len(misclassified_df)}")
    print(misclassified_df.head())

    # Optional: Heatmaps
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, cm, title in zip(axes, 
                             [confusion_matrix(y_true1, y_pred1),
                              confusion_matrix(y_true2, y_pred2),
                              confusion_matrix(y_true3, y_pred3)],
                             ["Level 1", "Level 2", "Level 3"]):
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_title(f"{title} Confusion Matrix")
    plt.tight_layout()
    plt.show()

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
df = pd.read_csv("/kaggle/input/task1-sentifeature/youtube_train_senti.csv")
df.rename(columns={
    "comment": "text",
    "Level 1": "level1",
    "Level 2": "level2",
    "Level 3": "level3"
}, inplace=True)
(y1, p1, y2, p2, y3, p3, misclassified_df,model) = train_and_evaluate(df, tokenizer)
print_results(y1, p1, y2, p2, y3, p3, misclassified_df)


====== Fold 1 ======


2025-07-04 08:19:52.048044: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751617192.293300      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751617192.371264      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Epoch 1:   0%|          | 0/254 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

...


📊 Epoch 2 Evaluation:
  Level 1 Accuracy: 0.8886, F1: 0.8684, Score: 1.7570
  Level 2 Accuracy: 0.8112, F1: 0.7854
  Level 3 Accuracy: 0.9317, F1: 0.9284


Epoch 3: 100%|██████████| 254/254 [00:51<00:00,  4.94it/s]



📊 Epoch 3 Evaluation:
  Level 1 Accuracy: 0.9198, F1: 0.9168, Score: 1.8366
  Level 2 Accuracy: 0.8112, F1: 0.7899
  Level 3 Accuracy: 0.9122, F1: 0.9099
💾 Best model saved at Epoch 3 with score = 1.8366

===== Level 1 Results =====
              precision    recall  f1-score   support

       NOISE       0.77      0.70      0.74       707
   OBJECTIVE       0.00      0.00      0.00        29
  SUBJECTIVE       0.94      0.96      0.95      3763

    accuracy                           0.91      4499
   macro avg       0.57      0.55      0.56      4499
weighted avg       0.91      0.91      0.91      4499

[[ 495    0  212]
 [   0    0   29]
 [ 144    0 3619]]

===== Level 2 Results (SUBJECTIVE only) =====
              precision    recall  f1-score   support

     NEUTRAL       0.84      0.89      0.87      2160
    NEGATIVE       0.80      0.82      0.81      1417
    POSITIVE       0.35      0.04      0.07       186

    accuracy                           0.82      3763
   macro av

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


ValueError: Number of classes, 3, does not match size of target_names, 4. Try specifying the labels parameter

In [5]:
# model1 = HierarchicalBertweet()
# model1.load_state_dict(torch.load("/kaggle/working/best_model_fold9.pt"))
# model1.cuda()
test_df = pd.read_csv("/kaggle/input/task1-sentifeature/youtube_test_senti.csv")
test_df.rename(columns={
    "comment": "text",
    "Level 1": "level1",
    "Level 2": "level2",
    "Level 3": "level3"
}, inplace=True)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from collections import Counter

def print_classwise_accuracy(y_true, y_pred, label_names, level_name):
    cm = confusion_matrix(y_true, y_pred)
    correct_per_class = cm.diagonal()
    total_per_class = cm.sum(axis=1)
    print(f"\nClass-wise Accuracy for {level_name}:")
    for i, label in enumerate(label_names):
        acc = 100.0 * correct_per_class[i] / total_per_class[i] if total_per_class[i] > 0 else 0.0
        print(f"  {label:<20} : {acc:.2f}% ({correct_per_class[i]}/{total_per_class[i]})")
        
def evaluate_on_test(model, test_df, tokenizer, batch_size=16):
    test_dataset = OpinionDataset(test_df['text'].tolist(),
                                   test_df['level1'].tolist(),
                                   test_df['level2'].tolist(),
                                   test_df['level3'].tolist(),
                                   test_df['senti_score'].tolist(),
                                   tokenizer)
    
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    model.eval()
    model.cuda()
    
    y_true1, y_pred1 = [], []
    y_true2, y_pred2 = [], []
    y_true3, y_pred3 = [], []

    label1_names = ["NOISE", "OBJECTIVE", "SUBJECTIVE"]
    label2_names = ["NEUTRAL", "NEGATIVE", "POSITIVE"]
    label3_names = ["NEUTRAL SENTIMENT", "QUESTIONS", "MISCELLANEOUS"]
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].cuda()
            attention_mask = batch['attention_mask'].cuda()
            label1 = batch['label1']
            label2 = batch['label2']
            label3 = batch['label3']
            senti_score = batch['senti_score'].cuda()

            out1, out2, out3 = model(input_ids, attention_mask, senti_score)
            pred1 = torch.argmax(out1, dim=1).cpu()
            pred2 = torch.argmax(out2, dim=1).cpu()
            pred3 = torch.argmax(out3, dim=1).cpu()

            y_true1.extend(label1.tolist())
            y_pred1.extend(pred1.tolist())

            for i in range(len(label1)):
                if label1[i] == 2:
                    y_true2.append(label2[i].item())
                    y_pred2.append(pred2[i].item())
                    
                    if label2[i] == 0:
                        y_true3.append(label3[i].item())
                        y_pred3.append(pred3[i].item())
    
    print("\n===== Level 1 Test Results =====")
    print("Accuracy:", accuracy_score(y_true1, y_pred1))
    print("F1 Score:", f1_score(y_true1, y_pred1, average='weighted'))
    print("Confusion Matrix:\n", confusion_matrix(y_true1, y_pred1))
    print_classwise_accuracy(y_true1, y_pred1, label1_names, "Level 1")
    
    print("\nLevel 2 Accuracy (on Level 1=2):", accuracy_score(y_true2, y_pred2))
    print("Level 2 F1 Score:", f1_score(y_true2, y_pred2, average='weighted'))
    print("Confusion Matrix - Level 2:")
    print(confusion_matrix(y_true2, y_pred2))
    print_classwise_accuracy(y_true2, y_pred2, label2_names, "Level 2")

    print("\nLevel 3 Accuracy (on Level 2=0):", accuracy_score(y_true3, y_pred3))
    print("Level 3 F1 Score:", f1_score(y_true3, y_pred3, average='weighted'))
    print("Confusion Matrix - Level 3:")
    print(confusion_matrix(y_true3, y_pred3))
    print_classwise_accuracy(y_true3, y_pred3, label3_names, "Level 3")
evaluate_on_test(model, test_df, tokenizer)


===== Level 1 Test Results =====
Accuracy: 0.904
F1 Score: 0.8989611244776498
Confusion Matrix:
 [[ 52   0  27]
 [  0   0   3]
 [ 18   0 400]]

Class-wise Accuracy for Level 1:
  NOISE                : 65.82% (52/79)
  OBJECTIVE            : 0.00% (0/3)
  SUBJECTIVE           : 95.69% (400/418)

Level 2 Accuracy (on Level 1=2): 0.8492822966507177
Level 2 F1 Score: 0.8331071365926388
Confusion Matrix - Level 2:
[[220  19   1]
 [ 21 134   2]
 [ 10  10   1]]

Class-wise Accuracy for Level 2:
  NEUTRAL              : 91.67% (220/240)
  NEGATIVE             : 85.35% (134/157)
  POSITIVE             : 4.76% (1/21)

Level 3 Accuracy (on Level 2=0): 0.9375
Level 3 F1 Score: 0.9355286738351255
Confusion Matrix - Level 3:
[[132   7   0]
 [  7  93   0]
 [  1   0   0]]

Class-wise Accuracy for Level 3:
  NEUTRAL SENTIMENT    : 94.96% (132/139)
  QUESTIONS            : 93.00% (93/100)
  MISCELLANEOUS        : 0.00% (0/1)


In [6]:
torch.save(model, 'tak1+k-fold+feature-youtube.pth')

In [18]:
test_df = pd.read_csv("/kaggle/input/task1-sentifeature/youtube_test_senti.csv")
test_df.rename(columns={
    "comment": "text",
    "Level 1": "level1",
    "Level 2": "level2",
    "Level 3": "level3"
}, inplace=True)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from collections import Counter

def print_classwise_accuracy(y_true, y_pred, label_names, level_name):
    cm = confusion_matrix(y_true, y_pred)
    correct_per_class = cm.diagonal()
    total_per_class = cm.sum(axis=1)
    print(f"\nClass-wise Accuracy for {level_name}:")
    for i, label in enumerate(label_names):
        acc = 100.0 * correct_per_class[i] / total_per_class[i] if total_per_class[i] > 0 else 0.0
        print(f"  {label:<20} : {acc:.2f}% ({correct_per_class[i]}/{total_per_class[i]})")
        
def evaluate_on_test(model, test_df, tokenizer, batch_size=16):
    test_dataset = OpinionDataset(test_df['text'].tolist(),
                                   test_df['level1'].tolist(),
                                   test_df['level2'].tolist(),
                                   test_df['level3'].tolist(),
                                   test_df['senti_score'].tolist(),
                                   tokenizer)
    
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    model.eval()
    model.cuda()
    
    y_true1, y_pred1 = [], []
    y_true2, y_pred2 = [], []
    y_true3, y_pred3 = [], []

    label1_names = ["NOISE", "OBJECTIVE", "SUBJECTIVE"]
    label2_names = ["NEUTRAL", "NEGATIVE", "POSITIVE"]
    label3_names = ["NEUTRAL SENTIMENT", "QUESTIONS", "MISCELLANEOUS"]
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].cuda()
            attention_mask = batch['attention_mask'].cuda()
            label1 = batch['label1']
            label2 = batch['label2']
            label3 = batch['label3']
            senti_score = batch['senti_score'].cuda()

            out1, out2, out3 = model(input_ids, attention_mask, senti_score)
            pred1 = torch.argmax(out1, dim=1).cpu()
            pred2 = torch.argmax(out2, dim=1).cpu()
            pred3 = torch.argmax(out3, dim=1).cpu()

            y_true1.extend(label1.tolist())
            y_pred1.extend(pred1.tolist())

            for i in range(len(label1)):
                if label1[i] == 2:
                    y_true2.append(label2[i].item())
                    y_pred2.append(pred2[i].item())
                    
                    if label2[i] == 0:
                        y_true3.append(label3[i].item())
                        y_pred3.append(pred3[i].item())
    
    print("\n===== Level 1 Test Results =====")
    print("Accuracy:", accuracy_score(y_true1, y_pred1))
    print("F1 Score:", f1_score(y_true1, y_pred1, average='weighted'))
    print("Confusion Matrix:\n", confusion_matrix(y_true1, y_pred1))
    print_classwise_accuracy(y_true1, y_pred1, label1_names, "Level 1")
    
    print("\nLevel 2 Accuracy (on Level 1=2):", accuracy_score(y_true2, y_pred2))
    print("Level 2 F1 Score:", f1_score(y_true2, y_pred2, average='weighted'))
    print("Confusion Matrix - Level 2:")
    print(confusion_matrix(y_true2, y_pred2))
    print_classwise_accuracy(y_true2, y_pred2, label2_names, "Level 2")

    print("\nLevel 3 Accuracy (on Level 2=0):", accuracy_score(y_true3, y_pred3))
    print("Level 3 F1 Score:", f1_score(y_true3, y_pred3, average='weighted'))
    print("Confusion Matrix - Level 3:")
    print(confusion_matrix(y_true3, y_pred3))
    print_classwise_accuracy(y_true3, y_pred3, label3_names, "Level 3")

model1 = HierarchicalBertweet()
model1.load_state_dict(torch.load("/kaggle/working/best_model_fold6.pt"))
model1.cuda()
evaluate_on_test(model1, test_df, tokenizer)


===== Level 1 Test Results =====
Accuracy: 0.922
F1 Score: 0.9194965949820789
Confusion Matrix:
 [[ 62   0  17]
 [  0   0   3]
 [ 19   0 399]]

Class-wise Accuracy for Level 1:
  NOISE                : 78.48% (62/79)
  OBJECTIVE            : 0.00% (0/3)
  SUBJECTIVE           : 95.45% (399/418)

Level 2 Accuracy (on Level 1=2): 0.8492822966507177
Level 2 F1 Score: 0.8294554136482298
Confusion Matrix - Level 2:
[[218  20   2]
 [ 20 137   0]
 [ 13   8   0]]

Class-wise Accuracy for Level 2:
  NEUTRAL              : 90.83% (218/240)
  NEGATIVE             : 87.26% (137/157)
  POSITIVE             : 0.00% (0/21)

Level 3 Accuracy (on Level 2=0): 0.9416666666666667
Level 3 F1 Score: 0.9397180762852405
Confusion Matrix - Level 3:
[[132   7   0]
 [  6  94   0]
 [  1   0   0]]

Class-wise Accuracy for Level 3:
  NEUTRAL SENTIMENT    : 94.96% (132/139)
  QUESTIONS            : 94.00% (94/100)
  MISCELLANEOUS        : 0.00% (0/1)


In [19]:
test_df = pd.read_csv("/kaggle/input/task1-sentifeature/CRYPTO_YOUTUBE_TEST_senti.csv")
texts = test_df['MAIN'].tolist()

class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts, senti_scores, tokenizer, max_len=128):  # Modified
        self.texts = texts
        self.senti_scores = senti_scores  # New
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.texts[idx],
                                padding='max_length',
                                truncation=True,
                                max_length=self.max_len,
                                return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'senti_score': torch.tensor(self.senti_scores[idx], dtype=torch.float)  # New
        }
        
test_dataset = InferenceDataset(texts, test_df['senti_score'].tolist(), tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16)

preds1, preds2, preds3 = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids'].cuda()
        attention_mask = batch['attention_mask'].cuda()
        senti_score = batch['senti_score'].cuda()  

        out1, out2, out3 = model1(input_ids, attention_mask, senti_score)

        p1 = torch.argmax(out1, dim=1).cpu().tolist()
        p2 = torch.argmax(out2, dim=1).cpu().tolist()
        p3 = torch.argmax(out3, dim=1).cpu().tolist()

        preds1.extend(p1)
        preds2.extend(p2)
        preds3.extend(p3)
        
test_df["Level 1"] = preds1
test_df["Level 2"] = preds2
test_df["Level 3"] = preds3

test_df[["comment_id","MAIN","Level 1", "Level 2", "Level 3"]].to_csv("k-fold6_senti_crypto_test_youtube.csv", index=False)

Predicting: 100%|██████████| 32/32 [00:01<00:00, 16.89it/s]
